# Run v4 training on Google Colab

This notebook contains ready-to-run cells to mount Google Drive, install dependencies, and start the big `train.py` run using GPU.

Notes:
- Upload the `v4` folder to your Drive (e.g. `/content/drive/MyDrive/v4`) or push `v4` to GitHub and clone it.
- Point `LOG_DIR` to a Drive path so checkpoints persist.
- Colab sessions can time out; consider Colab Pro/Pro+ or a cloud VM for very long runs.

In [ ]:
# 1) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

If you uploaded the `v4` folder to Drive, copy it into the local workspace.

In [ ]:
# Option A: copy v4 from Drive (update the path if needed)
!cp -r /content/drive/MyDrive/v4 /content/v4 || true
%ls -la /content/v4 || true

Or clone from GitHub (Option B). Replace the repo URL below with your repo that contains the `v4` folder.

In [ ]:
# Option B: clone from GitHub (uncomment and edit)
# !git clone https://github.com/yourname/yourrepo.git /content/repo
# !cp -r /content/repo/v4 /content/v4
%ls -la /content/v4 || true

Install dependencies. If you need a CUDA-enabled PyTorch, install the appropriate wheel after checking the CUDA version.

In [ ]:
# Upgrade pip and install requirements
!pip install --upgrade pip
!pip install -r /content/v4/requirements.txt
# Check GPU & driver
!nvidia-smi || true
import torch
print('torch.cuda.is_available():', torch.cuda.is_available())
print('torch version:', torch.__version__, 'cuda:', torch.version.cuda)

If `torch.cuda.is_available()` is False but `nvidia-smi` shows a GPU, install the matching CUDA wheel. Example for CUDA 11.8:

In [ ]:
# Example (edit/remove if not needed): install CUDA-enabled PyTorch wheel
# !pip install --index-url https://download.pytorch.org/whl/cu118 torch torchvision torchaudio --upgrade
# After installing, re-check:
# import torch; print(torch.cuda.is_available())

Start the large training using GPU 0 and save logs/checkpoints to Drive. Edit `LOG_DIR` to a Drive path you want.

In [ ]:
# Configure these variables before running:
LOG_DIR='/content/drive/MyDrive/tinystories_logs/v4'
CONFIG='/content/v4/configs/tinystories_grow_all.yaml'
MAX_ITER=4000
NUM_TASKS=2
mkdir -p $LOG_DIR
# Run in background with nohup so the notebook cell returns immediately; stdout/stderr go to the log file.
!nohup nice -n 10 ionice -c3 python /content/v4/train.py --config $CONFIG --num-tasks $NUM_TASKS --max-iter $MAX_ITER --gpu 0 --log_dir $LOG_DIR > $LOG_DIR/train.log 2>&1 & echo $! > $LOG_DIR/train.pid
print('Started training, logs ->', LOG_DIR + '/train.log')
!ls -la $LOG_DIR

Resume from checkpoint (run in a new cell when needed).

In [ ]:
# Example resume command (adjust paths)
# python /content/v4/train.py --config $CONFIG --resume --log_dir $LOG_DIR --gpu 0

Helpful monitoring commands (run in notebook cells):

In [ ]:
# Tail training log
!tail -n 200 $LOG_DIR/train.log || true
# Show PID
!cat $LOG_DIR/train.pid || true
# GPU usage
!nvidia-smi || true